# Prepare matrices for SCENICplus

In [4]:
###################
## Load packages
###################
suppressPackageStartupMessages({
    library(scran)
    library(scater)
    library(Seurat)
    library(ArchR)
})

here::i_am("revisions/05_SCENICplus/from_scratch/00_prepare_matrices.ipynb")

source(here::here("settings.R"))
source(here::here("utils.R"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code

Warning message:
“package ‘rtracklayer’ was built under R version 4.2.3”


In [5]:
###################
## I/O
###################

# Metadata
args$metadata = file.path(io$basedir, 'results/rna_atac/clustering/metadata_celltype_annotated_v2.txt.gz')

# RNA_sce
args$rna_sce = file.path(io$basedir, 'processed/rna/SingleCellExperiment.rds')

# Archr
args$archr_directory = file.path(io$basedir, 'processed/atac/archR')

# output
args$outdir = file.path("/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/revisions/05_SCENICplus/SCENICplus/")
dir.create(args$outdir, recursive=TRUE, showWarnings =FALSE)

In [6]:
meta = fread(args$metadata) %>% 
    .[genotype=='WT']

In [8]:
archrvitro = loadArchRProject(args$archr_directory)[meta$cell]

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [9]:
getAvailableMatrices(archrvitro)

[1] "DeviationMatrix_CISBP"   "DeviationMatrix_JASPAR" 
 [3] "GeneScoreMatrix_TSS"     "GeneScoreMatrix_distal" 
 [5] "MarkerPeaksMatrix"       "MotifMatrix"            
 [7] "PeakMatrix"              "PeakMatrix_atlas"       
 [9] "TileMatrix"              "silicoChIP_CISBPMatrix" 
[11] "silicoChIP_CISBP_v2"     "silicoChIP_JASPARMatrix"
[13] "silicoChIP_JASPAR_v2"

In [10]:
addArchRThreads(32)
atac.sce = getMatrixFromProject(archrvitro, useMatrix = 'PeakMatrix')

Setting default number of Parallel threads to 32.

ArchR logging to : ArchRLogs/ArchR-getMatrixFromProject-5a9599b58ad-Date-2024-11-26_Time-13-53-24.log
If there is an issue, please report to github with logFile!

2024-11-26 13:54:25 : Organizing colData, 1.025 mins elapsed.

2024-11-26 13:54:26 : Organizing rowData, 1.033 mins elapsed.

2024-11-26 13:54:26 : Organizing rowRanges, 1.033 mins elapsed.

2024-11-26 13:54:26 : Organizing Assays (1 of 1), 1.033 mins elapsed.

2024-11-26 13:54:51 : Constructing SummarizedExperiment, 1.452 mins elapsed.

2024-11-26 13:55:15 : Finished Matrix Creation, 1.859 mins elapsed.



In [11]:
######
## subset to shared cells
######

# Load RNA SingleCellExperiment
rna.sce <- readRDS(args$rna_sce)
#rna.sce = rna.sce[,match(meta$cell, colnames(rna.sce))]
# Make sure that samples are consistent
cells <- intersect(colnames(rna.sce),colnames(atac.sce))
rna.sce <- rna.sce[,cells]
atac.sce <- atac.sce[,cells]

In [12]:
rna.sce
atac.sce

class: SingleCellExperiment 
dim: 32285 19740 
metadata(0):
assays(1): counts
rownames(32285): Xkr4 Gm1992 ... AC234645.1 AC149090.1
rowData names(0):
colnames(19740): rv_eo_deg_day3_5_control#AAACAGCCAAACCTTG-1
  rv_eo_deg_day3_5_control#AAACAGCCAGCAACCT-1 ...
  2_Eo_DEG_G9_day5_VC#TTTGTCCCATTAAGTC-1
  2_Eo_DEG_G9_day5_VC#TTTGTTGGTACCGGAT-1
colData names(12): barcode sample ... pass_rnaQC sizeFactor
reducedDimNames(0):
mainExpName: RNA
altExpNames(0):

class: RangedSummarizedExperiment 
dim: 234908 19740 
metadata(0):
assays(1): PeakMatrix
rownames: NULL
rowData names(1): idx
colnames(19740): rv_eo_deg_day3_5_control#AAACAGCCAAACCTTG-1
  rv_eo_deg_day3_5_control#AAACAGCCAGCAACCT-1 ...
  2_Eo_DEG_G9_day5_VC#TTTGTCCCATTAAGTC-1
  2_Eo_DEG_G9_day5_VC#TTTGTTGGTACCGGAT-1
colData names(39): BlacklistRatio nDiFrags ... FRIP replicate

In [13]:
atac.sce = as(atac.sce, 'SingleCellExperiment')

In [16]:
vitroPeaks = archrvitro@peakSet

In [18]:
rownames(atac.sce) = paste0(seqnames(vitroPeaks), ':', start(vitroPeaks), '-', end(vitroPeaks))

In [19]:
bed = data.frame(chr = seqnames(vitroPeaks),
                 start = start(vitroPeaks), 
                 end = end(vitroPeaks))

In [20]:
fwrite(bed, col.names = F, file.path(args$outdir, 'regions.bed'), sep = '\t')

In [22]:
# Save metadata
fwrite(meta, file.path(args$outdir, 'meta_scenic.txt'))

In [24]:
## Save output for SCENIC+
write.table(colnames(rna.sce), file.path(args$outdir, 'cells.txt'))
write.table(rownames(rna.sce), file.path(args$outdir, 'genes.txt'))
write.table(rownames(atac.sce), file.path(args$outdir, 'peaks.txt'), sep = '\t', row.names = F)

# Save RNA as mtx file
writeMM(as(counts(rna.sce), "dgCMatrix"), file=file.path(args$outdir, 'rna.mtx'))
assayNames(atac.sce) = 'counts'
writeMM(as(counts(atac.sce), "dgCMatrix"), file=file.path(args$outdir, 'atac.mtx'))


NULL

NULL